In [ ]:
# ==========================================================
# Cyberwarfare & Espionage ETL Dashboard  (Colab + VPS Ready)
# Author: Duncan Lawrence
# ==========================================================

# --- Section 1. Setup ---
!pip install pandas plotly leafmap flask geopandas pycountry --quiet

import pandas as pd
import plotly.express as px
import leafmap.foliumap as leafmap
import os, pycountry

# Detect environment
run_mode = "colab" if "COLAB_RELEASE_TAG" in os.environ else "vps"
print("Running in:", run_mode.upper())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.9/632.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.0 MB/s eta 0:00:00
Running in: COLAB


In [ ]:
# --- Section 2. Extract + Transform ---

# Load DrSufi's CyberData dataset
url = "https://raw.githubusercontent.com/DrSufi/CyberData/main/cyber_data.csv"
df = pd.read_csv(url)

print("Dataset loaded:", df.shape)
print("Columns:", list(df.columns))
df.head()

# Clean column names
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Rename key columns for clarity
df = df.rename(columns={"attackdate": "date", "country": "country"})

# Convert dates to datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year

# Keep the main columns of interest
metric_cols = [c for c in df.columns if not c.startswith("rank") and c not in ["date", "country", "year"]]
rank_cols = [c for c in df.columns if c.startswith("rank")]

print(f"Metrics: {metric_cols}")
print(f"Rankings: {rank_cols[:3]} ...")

# Drop rows without country or date
df = df.dropna(subset=["country", "year"])

# Group by country/year to get average threat activity
agg = df.groupby(["country", "year"])[metric_cols].mean().reset_index()

print("Aggregated rows:", len(agg))
agg.head()


Dataset loaded: (77623, 18)
Columns: ['AttackDate', 'Country', 'Spam', 'Ransomware', 'Local Infection', 'Exploit', 'Malicious Mail', 'Network Attack', 'On Demand Scan', 'Web Threat', 'Rank Spam', 'Rank Ransomware', 'Rank Local Infection', 'Rank Exploit', 'Rank Malicious Mail', 'Rank Network Attack', 'Rank On Demand Scan', 'Rank Web Threat']
Metrics: ['spam', 'ransomware', 'local_infection', 'exploit', 'malicious_mail', 'network_attack', 'on_demand_scan', 'web_threat']
Rankings: ['rank_spam', 'rank_ransomware', 'rank_local_infection'] ...
Aggregated rows: 433


,country,year,spam,ransomware,local_infection,exploit,malicious_mail,network_attack,on_demand_scan,web_threat
0,Anguilla,2022.0,0.000020,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Anguilla,2023.0,0.000010,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Antarctica,2023.0,0.000260,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Antigua and Barbuda,2022.0,0.000017,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Antigua and Barbuda,2023.0,0.000060,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# --- Section 3. Load ---
# Save cleaned dataset locally (optional)
df.to_csv("cyber_incidents_clean.csv", index=False)
print(f"Rows after cleaning: {len(df)}")


Rows after cleaning: 31151


In [ ]:
# ==============================================================
#  Section 4. Visualization Suite (self-contained, rerunnable)
# ==============================================================

# ---- 4a. Plotly Bar Chart: Average Threat Level by Type ----
import plotly.express as px
import pandas as pd

# make sure 'agg' exists
if "agg" not in locals():
    from datetime import datetime
    # reload dataset quickly if notebook restarted
    url = "https://raw.githubusercontent.com/DrSufi/CyberData/main/cyber_data.csv"
    raw = pd.read_csv(url)
    raw.columns = [c.lower().replace(" ", "_") for c in raw.columns]
    raw["date"] = pd.to_datetime(raw["attackdate"], errors="coerce")
    raw["year"] = raw["date"].dt.year
    metric_cols = [c for c in raw.columns if c.startswith(("spam", "ransomware","local","exploit","malicious","network","on","web"))]
    agg = raw.groupby(["country","year"])[metric_cols].mean().reset_index()

metric_cols = [c for c in agg.columns if c not in ["country","year"]]

fig = px.bar(
    agg.melt(id_vars=["country","year"], var_name="threat_type", value_name="intensity"),
    x="year", y="intensity", color="threat_type",
    title="Average Cyber Threat Levels Over Time (Global Aggregates)",
)
fig.update_layout(template="plotly_white", width=900, height=500)
fig.show()

# ---- 4b. Top 10 Countries by Combined Threat Activity ----
agg["total_threat_score"] = agg[metric_cols].sum(axis=1)
latest_year = int(agg["year"].max())
top_countries = agg[agg["year"] == latest_year].nlargest(10, "total_threat_score")

fig2 = px.bar(
    top_countries.sort_values("total_threat_score"),
    x="total_threat_score", y="country",
    orientation="h",
    title=f"Top 10 Countries by Threat Activity ({latest_year})",
    color="total_threat_score",
)
fig2.update_layout(template="plotly_white", height=500)
fig2.show()

# --- Section 4c. Leafmap Heat Visualization (Final Colab-Compatible) ---
import folium  # leafmap uses folium underneath
from leafmap import foliumap as leafmap
import pycountry, random, re

# recreate map object
m = leafmap.Map(center=[20, 0], zoom=2)

def get_country_coords(country_name):
    """Return centroid lat/lon for a given country name with multiple fallback methods."""
    name = country_name.lower().strip()
    fallback_coords = {
        "people's republic of china": (35.0, 103.0),
        "russian federation": (61.0, 99.0),
        "united states of america": (39.8, -98.6),
        "kingdom of the netherlands": (52.3, 5.3),
        "people's republic of bangladesh": (23.7, 90.3),
        "socialist republic of vietnam": (16.5, 107.0),
        "republic of the union of myanmar": (21.0, 96.0),
        "people's democratic republic of algeria": (28.0, 2.6),
        "islamic republic of afghanistan": (34.0, 66.0),
        "republic of kazakhstan": (48.0, 67.0),
    }
    if name in fallback_coords:
        return fallback_coords[name]
    clean = re.sub(r"(?i)(people's republic of|republic of|kingdom of|socialist|islamic|union of|democratic|state of|federal|the)", "", name).strip()
    try:
        c = pycountry.countries.lookup(clean)
        bounds = leafmap.country_bounds(c.alpha_3)
        lon = (bounds[0][0] + bounds[1][0]) / 2
        lat = (bounds[0][1] + bounds[1][1]) / 2
        return lat, lon
    except Exception:
        return (None, None)

# generate distinct colors
colors = [f"#{random.randint(40,255):02x}{random.randint(40,255):02x}{random.randint(40,255):02x}"
          for _ in range(len(top_countries))]

heat_data = []

# add Folium circle markers directly
for idx, row in enumerate(top_countries.itertuples()):
    lat, lon = get_country_coords(row.country)
    if lat and lon:
        heat_data.append([lat, lon, row.total_threat_score])
        folium.CircleMarker(
            location=[lat, lon],
            radius=8,
            color=colors[idx],
            fill=True,
            fill_opacity=0.6,
            popup=f"{row.country}: {row.total_threat_score:.1f}"
        ).add_to(m)

# add heatmap and legend
m.add_heatmap(heat_data, name="Threat Intensity Heatmap", radius=35)
m.add_layer_control()
m

In [ ]:
# --- Section 4a. Plotly Bar Chart: Average Threat Level by Type ---
import plotly.express as px

fig = px.bar(
    agg.melt(id_vars=["country", "year"], var_name="threat_type", value_name="intensity"),
    x="year", y="intensity", color="threat_type",
    title="Average Cyber Threat Levels Over Time (Global Aggregates)",
)
fig.update_layout(template="plotly_white", width=900, height=500)
fig.show()


In [ ]:
# --- Section 4b. Top 10 Countries by Combined Threat Activity ---
agg["total_threat_score"] = agg[metric_cols].sum(axis=1)
latest_year = agg["year"].max()
top_countries = agg[agg["year"] == latest_year].nlargest(10, "total_threat_score")

fig2 = px.bar(
    top_countries.sort_values("total_threat_score"),
    x="total_threat_score", y="country",
    orientation="h",
    title=f"Top 10 Countries by Threat Activity ({latest_year})",
    color="total_threat_score",
)
fig2.update_layout(template="plotly_white", height=500)
fig2.show()


In [ ]:
# --- Section 4c. Leafmap Heat Visualization Setup (Corrected) ---
!pip install leafmap geopandas shapely pycountry folium --quiet

# Import top-level leafmap first to check version
import leafmap
print("Leafmap version:", leafmap.__version__)

# Then import the folium backend (for Colab maps)
from leafmap import foliumap as leafmap
print("Folium backend loaded OK")

# Other imports
import pycountry
import random, re

# fallback coords for common political forms that pycountry misses
fallback_coords = {
    "people's republic of china": (35.0, 103.0),
    "russian federation": (61.0, 99.0),
    "united states of america": (39.8, -98.6),
    "kingdom of the netherlands": (52.3, 5.3),
    "people's republic of bangladesh": (23.7, 90.3),
    "socialist republic of vietnam": (16.5, 107.0),
    "republic of the union of myanmar": (21.0, 96.0),
    "people's democratic republic of algeria": (28.0, 2.6),
    "islamic republic of afghanistan": (34.0, 66.0),
    "republic of kazakhstan": (48.0, 67.0),
}

def get_country_coords(country_name):
    """Return centroid lat/lon for a given country name with multiple fallback methods."""
    name = country_name.lower().strip()
    # quick fallback for known tricky names
    if name in fallback_coords:
        return fallback_coords[name]

    # simplified lookup
    clean = re.sub(r"(?i)(people's republic of|republic of|kingdom of|socialist|islamic|union of|democratic|state of|federal|the)", "", name).strip()
    try:
        c = pycountry.countries.lookup(clean)
        bounds = leafmap.country_bounds(c.alpha_3)
        lon = (bounds[0][0] + bounds[1][0]) / 2
        lat = (bounds[0][1] + bounds[1][1]) / 2
        return lat, lon
    except Exception:
        return (None, None)

# create map
m = leafmap.Map(center=[20, 0], zoom=2)

# build color palette
colors = [(random.randint(60, 255), random.randint(60, 255), random.randint(60, 255))
          for _ in range(len(top_countries))]

# collect heatmap data
heat_data = []

for (idx, row) in enumerate(top_countries.itertuples()):
    lat, lon = get_country_coords(row.country)
    if lat and lon:
        heat_data.append([lat, lon, row.total_threat_score])
        # add circle markers
        m.add_circle_marker(
            location=[lat, lon],
            radius=12,
            color=f"rgb{colors[idx]}",
            fill=True,
            fill_opacity=0.6,
            popup=f"{row.country}: {row.total_threat_score:.1f}"
        )

# add legend
m.add_legend(
    title=f"Top Cyber Threat Countries ({latest_year})",
    labels=top_countries["country"].tolist(),
    colors=colors
)

# add toggleable heatmap layer
m.add_heatmap(heat_data, name="Threat Intensity Heatmap", radius=35)
m.add_layer_control()

m

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.9/632.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.4 MB/s eta 0:00:00
Leafmap version: 0.57.10
Folium backend loaded OK


NameError: name 'top_countries' is not defined

In [ ]:
# --- Section 4c. Leafmap Heat Visualization (Final Working Version) ---
import leafmap.foliumap as leafmap
import pycountry
import random

def get_country_coords(country_name):
    """Return approximate centroid lat/lon for a given country name."""
    try:
        c = pycountry.countries.lookup(country_name)
        bounds = leafmap.country_bounds(c.alpha_3)
        lon = (bounds[0][0] + bounds[1][0]) / 2
        lat = (bounds[0][1] + bounds[1][1]) / 2
        return lat, lon
    except Exception:
        return (None, None)

# Initialize map
m = leafmap.Map(center=[20, 0], zoom=2)

# Add markers for top countries
for _, row in top_countries.iterrows():
    lat, lon = get_country_coords(row["country"])
    if lat and lon:
        m.add_marker(
            location=[lat, lon],
            popup=f"{row['country']}: {row['total_threat_score']:.1f}",
            icon="shield",
            draggable=False
        )

# Generate random distinguishable RGB tuples for each label
colors = [(random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
          for _ in range(len(top_countries))]

# Add legend
m.add_legend(
    title=f"Top Cyber Threat Countries ({latest_year})",
    labels=top_countries["country"].tolist(),
    colors=colors
)

m



In [ ]:
# --- Section 5. Flask Export (for VPS use) ---
# To deploy on Hostinger VPS, save this as app.py in the same directory.

flask_example = r"""
from flask import Flask, render_template_string
import pandas as pd, plotly.express as px

app = Flask(__name__)

@app.route('/')
def dashboard():
    df = pd.read_csv('cyber_incidents_clean.csv')
    fig = px.histogram(df, x='year', color='origin_country',
                       title='Cyber Incidents by Year and Origin Country')
    html = fig.to_html(full_html=False)
    return render_template_string('<h1>Cyber Dashboard</h1>' + html)

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8080)
"""
print("Save the text below as app.py for VPS deployment:\n")
print(flask_example)